In [3]:
%pip install openai

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [9]:
from openai import OpenAI

BASE_URL = "https://gpustack.cds.utsa.edu/v1"
API_KEY = "REPLACE-WITH-YOUR-KEY"

client = OpenAI(
    base_url=BASE_URL,
    api_key=API_KEY
)

# Ask the server which model it is running
#MODEL = client.models.list().data[0].id

#print("Connected. Model:", MODEL)

models = client.models.list().data

for model in models:
    print(model.id)

qwen3.5-122b-a10b
qwen3-vl-8b-instruct
qwen3-vl-embedding-8b
qwen3-vl-32b-instruct
qwen3-coder-next


In [11]:
import time

def ask_model(prompt: str, temperature: float = 1.0, retries: int = 2):
    """Send one prompt to Qwen Coder and return its text reply, retrying on failure."""

    for attempt in range(retries + 1):
        try:
            resp = client.chat.completions.create(
                model=models[0].id,
                messages=[{"role": "user", "content": prompt}],
                temperature=temperature,
                top_p=0.95,
            )

            return resp.choices[0].message.content

        except Exception:
            if attempt == retries:
                raise

            time.sleep(1.5 * (attempt + 1))
print(ask_model("Say hello in one short sentence."))



Hello there!


In [12]:
question = (
    "I am writing a Python function triage_log(line) that asks an LLM to classify a "
    "security log line and returns a dataclass with label and reason. Suggest a robust "
    "prompt format and a defensive way to parse the reply. Keep it concise."
)

print(ask_model(question))



```python
from dataclasses import dataclass
from typing import Literal
import json
import re

# 1. Strict Schema Definition
@dataclass
class TriageResult:
    label: Literal["safe", "suspicious", "malicious"]
    reason: str

    @classmethod
    def validate(cls, d: dict) -> "TriageResult":
        # Defensive allowlist check
        if d.get("label") not in {"safe", "suspicious", "malicious"}:
            raise ValueError("Invalid label")
        if not isinstance(d.get("reason"), str):
            raise ValueError("Reason must be string")
        return cls(**d)

# 2. Robust Prompt
SYSTEM_PROMPT = """You are a security log classifier.
Output ONLY a raw JSON object: {"label": "safe|suspicious|malicious", "reason": "string"}.
No markdown, no explanations, no other text."""

def triage_log(line: str, client: Any) -> TriageResult:
    # 3. Defensive Call & Parsing
    try:
        # Ensure line is string and safe for prompt context
        clean_line = str(line).replace("\n", " ").str

In [13]:
from dataclasses import dataclass
from enum import Enum
import re

class Label(str, Enum):
    BENIGN = "BENIGN"
    SUSPICIOUS = "SUSPICIOUS"
    MALICIOUS = "MALICIOUS"
    UNKNOWN = "UNKNOWN"

@dataclass
class Triage:
    line: str
    label: Label
    reason: str

LABEL_RE = re.compile(r"^LABEL:\s*(\w+)", re.IGNORECASE | re.MULTILINE)
REASON_RE = re.compile(r"^REASON:\s*(.+)", re.IGNORECASE | re.MULTILINE)

In [14]:
def triage_log(line: str) -> Triage:
    prompt = (
        "You are a security log triage assistant.\n"
        "Classify the log line as BENIGN, SUSPICIOUS, or MALICIOUS and give a one "
        "sentence reason.\n"
        "Reply in EXACTLY this format:\n"
        "LABEL: <BENIGN|SUSPICIOUS|MALICIOUS>\n"
        "REASON: <one sentence>\n\n"
        f"Log line:\n{line}\n"
    )

    text = ask_model(prompt)

    label_match = LABEL_RE.search(text)
    raw = label_match.group(1).upper() if label_match else "UNKNOWN"

    label = Label(raw) if raw in Label.__members__ else Label.UNKNOWN

    reason_match = REASON_RE.search(text)
    reason = reason_match.group(1).strip() if reason_match else text.strip()

    return Triage(line=line, label=label, reason=reason)

In [15]:
TOOL_NAME = "SentinelSystem"

In [16]:
print(triage_log("Failed password for root from 203.0.113.9 port 4022 ssh2"))

Triage(line='Failed password for root from 203.0.113.9 port 4022 ssh2', label=<Label.SUSPICIOUS: 'SUSPICIOUS'>, reason='This log entry indicates a failed authentication attempt for the privileged root account via SSH, which is a common indicator of brute-force activity requiring further investigation.')


In [21]:
LOG_FILE = "security_log.csv"  # Windows CSV
MAX_ENTRIES = 10

def read_log_lines(path, limit=MAX_ENTRIES):
    with open(path, encoding="utf-8", errors="ignore") as f:
        lines = [ln.strip() for ln in f if ln.strip()]

    return lines[:limit]


entries = read_log_lines(LOG_FILE)

results = [triage_log(line) for line in entries]

print(f"Assessed {len(results)} entries from {LOG_FILE}")

Assessed 10 entries from security_log.csv


In [22]:
from datetime import datetime

def write_report(results, tool_name, path="assessment.md"):
    with open(path, "w", encoding="utf-8") as f:
        f.write(f"# {tool_name} Security Log Assessment\n\n")
        f.write(f"Generated: {datetime.now():%Y-%m-%d %H:%M}\n\n")
        f.write(f"Entries assessed: {len(results)}\n\n")

        for r in results:
            f.write(f"- **{r.label.value}** {r.reason}\n")
            f.write(f"  - `{r.line}`\n")

    return path

In [23]:
report_path = write_report(results, TOOL_NAME)

print("Wrote", report_path, "\n")

print(open(report_path, encoding="utf-8").read())

Wrote assessment.md 

# SentinelSystem Security Log Assessment

Generated: 2026-11-18 19:47

Entries assessed: 10

- **BENIGN** This line is a header row containing column definitions instead of an actual security event.
  - `﻿Keywords,Date and Time,Source,Event ID,Task Category`
- **BENIGN** This is a standard Windows security audit event indicating a routine check of group membership that is commonly generated by legitimate administrative tools and system management software.
  - `Audit Success,11/18/2026 7:12:12 PM,Microsoft-Windows-Security-Auditing,4798,User Account Management,"A user's local group membership was enumerated.`
- **BENIGN** This entry is an incomplete log record containing only a label with no security-relevant data.
  - `Subject:`
- **BENIGN** This log entry indicates activity from the built-in SYSTEM account, which is standard for core Windows operating system processes.
  - `Security ID:		SYSTEM`
- **BENIGN** The trailing dollar sign indicates this is a standard 

In [5]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# IS2083 Lab 5: AI Automation Tool, Part 1\n",
    "\n",
    "**Before you begin, you need TWO things:**\n",
    "\n",
    "1. Your **Jupyter login** (you used it to open this notebook).\n",
    "2. Your **API key** for the class Qwen Coder model server (from your instructor).\n",
    "\n",
    "This lab assumes you already know basic Python. The new material is the **AI model**, its **API**, and **prompt design**. You will connect to the class **Qwen Coder** model, use it as a coding assistant, and build an **AI-powered security log assessment** tool. At the end you will feed it a real security log exported from **Windows** or **macOS** and have it write an assessment report. Run cells with **Shift + Enter**, top to bottom."
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 1: Install the openai client (run once)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "%pip install openai"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 2: Connect to the Qwen Coder model server\n",
    "\n",
    "Paste the `BASE_URL` (ends in `/v1`) and your `API_KEY` from your instructor. The code then queries the server for the served model id."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "from openai import OpenAI\n",
    "\n",
    "BASE_URL = \"Replace with GPUstack URL"   # class model server address\n",
    "API_KEY  = \"Replace with API Key"  # your personal API key\n",
    "\n",
    "client = OpenAI(base_url=BASE_URL, api_key=API_KEY)\n",
    "MODEL = client.models.list().data[1].id   # discover the served model id\n",
    "print(\"Connected. Model:\", MODEL)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 3: A reusable model helper (with retries)\n",
    "\n",
    "`ask_model` wraps one chat completion and retries on transient errors."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import time\n",
    "\n",
    "def ask_model(prompt: str, temperature: float = 0.2, retries: int = 2) -> str:\n",
    "    \"\"\"Send one prompt to Qwen Coder and return its text reply, retrying on failure.\"\"\"\n",
    "    for attempt in range(retries + 1):\n",
    "        try:\n",
    "            resp = client.chat.completions.create(\n",
    "                model=MODEL,\n",
    "                messages=[{\"role\": \"user\", \"content\": prompt}],\n",
    "                temperature=temperature,\n",
    "            )\n",
    "            return resp.choices[0].message.content\n",
    "        except Exception:\n",
    "            if attempt == retries:\n",
    "                raise\n",
    "            time.sleep(1.5 * (attempt + 1))\n",
    "\n",
    "print(ask_model(\"Say hello in one short sentence.\"))"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 4: Plan the tool\n",
    "\n",
    "Input: one security log entry. Output: a structured result with a `label` (BENIGN / SUSPICIOUS / MALICIOUS) and a one-sentence `reason`. You solve the structure problem with a strict output format in the prompt plus defensive parsing."
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 5: Use Qwen Coder as a coding assistant\n",
    "\n",
    "Ask the model to sketch an approach, then keep only what you understand."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "question = (\n",
    "    \"I am writing a Python function that asks an LLM to classify a security log entry \"\n",
    "    \"and returns a dataclass with label and reason. Suggest a robust prompt format and \"\n",
    "    \"a defensive way to parse the reply. Keep it concise.\"\n",
    ")\n",
    "print(ask_model(question))"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 6: Define the result type and parsers\n",
    "\n",
    "An `Enum` fixes the labels, a `dataclass` gives a typed result, and two regexes extract the fields."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "from dataclasses import dataclass\n",
    "from enum import Enum\n",
    "import re\n",
    "\n",
    "class Label(str, Enum):\n",
    "    BENIGN = \"BENIGN\"\n",
    "    SUSPICIOUS = \"SUSPICIOUS\"\n",
    "    MALICIOUS = \"MALICIOUS\"\n",
    "    UNKNOWN = \"UNKNOWN\"\n",
    "\n",
    "@dataclass\n",
    "class Triage:\n",
    "    line: str\n",
    "    label: Label\n",
    "    reason: str\n",
    "\n",
    "LABEL_RE = re.compile(r\"^LABEL:\\s*(\\w+)\", re.IGNORECASE | re.MULTILINE)\n",
    "REASON_RE = re.compile(r\"^REASON:\\s*(.+)\", re.IGNORECASE | re.MULTILINE)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 7: Name your tool\n",
    "\n",
    "This tool can go in your professional portfolio, so give it a name you would be proud to show. `triage_log` is a fine name for the function, but the **tool** itself can be called anything: Play it straight (SentinelScan, ThreatLens, LogHawk, WatchTower) or give it some personality (SnitchBot, LogFather, GrepReaper, Sherlock Logs, The Log Whisperer, LogNessMonster), your call. Set it here; it appears in your report and your repository name."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "TOOL_NAME = \"SentinelScan\"   # <-- change this to your own creative name"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 8: Build the tool\n",
    "\n",
    "Write the classifier: a strict prompt, one `ask_model` call, then defensive parsing that falls back to `Label.UNKNOWN`. (You may rename `triage_log` to match your tool if you like.)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "def triage_log(line: str) -> Triage:\n",
    "    # TODO 1: write a prompt that demands EXACTLY these two lines back:\n",
    "    #   LABEL: <BENIGN|SUSPICIOUS|MALICIOUS>\n",
    "    #   REASON: <one sentence>\n",
    "    prompt = f\"REPLACE ME\\n\\nLog entry:\\n{line}\\n\"\n",
    "\n",
    "    text = ask_model(prompt)\n",
    "\n",
    "    # TODO 2: parse with LABEL_RE and REASON_RE; fall back to Label.UNKNOWN\n",
    "    label = Label.UNKNOWN\n",
    "    reason = \"\"\n",
    "\n",
    "    return Triage(line=line, label=label, reason=reason)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 9: Quick sanity check\n",
    "Confirm the tool works on one entry before feeding it a file."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "print(triage_log(\"Failed password for root from 203.0.113.9 port 4022 ssh2\"))"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 10: Assess a real security log file\n",
    "\n",
    "Export your computer's security log to **one file** and upload it here. Do the ONE that matches your computer; you only turn in one (Windows or macOS).\n",
    "\n",
    "**Windows (Event Viewer):** open Event Viewer, expand **Windows Logs**, click **Security**. In the right panel click **Save All Events As...**, set *Save as type* to **CSV (Comma Separated)**, and save it as `security_log.csv`.\n",
    "\n",
    "**macOS (Terminal):** export recent security-related events to one file:\n",
    "\n",
    "```\n",
    "log show --predicate 'process == \"sudo\" OR process == \"sshd\" OR process == \"loginwindow\" OR eventMessage CONTAINS[c] \"authentication\"' --last 6h --style syslog > security_log.txt\n",
    "```\n",
    "\n",
    "If that file is nearly empty, use: `log show --last 1h --style syslog > security_log.txt`\n",
    "\n",
    "Then click **Upload** in the Jupyter file browser and add your `security_log.csv` (Windows) or `security_log.txt` (macOS) to this notebook's folder. Set `LOG_FILE` below to match."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "LOG_FILE = \"security_log.csv\"   # Windows CSV, or \"security_log.txt\" on macOS\n",
    "MAX_ENTRIES = 20                 # keep this first version small and quick\n",
    "\n",
    "def read_log_lines(path, limit=MAX_ENTRIES):\n",
    "    with open(path, encoding=\"utf-8\", errors=\"ignore\") as f:\n",
    "        lines = [ln.strip() for ln in f if ln.strip()]\n",
    "    return lines[:limit]\n",
    "\n",
    "entries = read_log_lines(LOG_FILE)\n",
    "results = [triage_log(line) for line in entries]\n",
    "print(f\"Assessed {len(results)} entries from {LOG_FILE}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 11: Write the assessment report\n",
    "\n",
    "Have your tool write a Markdown report named after your tool. This report is your deliverable."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "from datetime import datetime\n",
    "\n",
    "def write_report(results, tool_name, path=\"assessment.md\"):\n",
    "    with open(path, \"w\", encoding=\"utf-8\") as f:\n",
    "        f.write(f\"# {tool_name} Security Log Assessment\\n\\n\")\n",
    "        f.write(f\"Generated: {datetime.now():%Y-%m-%d %H:%M}\\n\\n\")\n",
    "        f.write(f\"Entries assessed: {len(results)}\\n\\n\")\n",
    "        for r in results:\n",
    "            f.write(f\"- **{r.label.value}** {r.reason}\\n\")\n",
    "            f.write(f\"  - `{r.line}`\\n\")\n",
    "    return path\n",
    "\n",
    "report_path = write_report(results, TOOL_NAME)\n",
    "print(\"Wrote\", report_path, \"\\n\")\n",
    "print(open(report_path, encoding=\"utf-8\").read())"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 12: Read, test, and trust\n",
    "\n",
    "Double-click to edit and answer briefly:\n",
    "\n",
    "1. Why model the output with an `Enum` and `dataclass` instead of a bare dict?\n",
    "2. What does your defensive parsing do when the model ignores the format?\n",
    "3. Which entries in your report look wrong, and how would you catch that?\n",
    "\n",
    "_Your answers:_"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 13: Save and submit\n",
    "\n",
    "1. **File > Save Notebook**, then **File > Download** for the `.ipynb`.\n",
    "2. Download your report `assessment.md` too.\n",
    "3. On GitHub, create a **public** repository **named after your tool** (for example `sentinelscan`). Do **not** name it lab5 or use the course name.\n",
    "4. **Add file > Upload files**: upload your notebook and `assessment.md`. Do **not** upload the raw security log; it may contain private data.\n",
    "5. Before uploading, **remove your real API key** from the connect cell.\n",
    "6. Take the screenshots listed in the handout into the submission template."
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.x"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}

NameError: name 'null' is not defined